In [1]:
#import + session DB + subgraph storage obj


from HOGDB.graph.subgraph import Subgraph  
from HOGDB.graph.subgraph import SubgraphEdge 
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase

# INIT DB

db = Neo4jDatabase()
graph = GraphwithSubgraphStorage(db)

print("YOU ARE SUCCESSFULLY CONNECTED TO THE ACTIVE DB ! ")

YOU ARE SUCCESSFULLY CONNECTED TO THE ACTIVE DB ! 


In [17]:
#IS1 


def IS1_profile(graph, personId: int):

    # (pr:Person)-[:IS_LOCATED_IN]->(pl:Place)
    tp= "city"
    pr = Node(labels=[Label("Person")])
    pl = Node(labels=[Label("Place")])
    e = Edge(pr, pl, label=Label("IS_LOCATED_IN"))
    

    path = Path()
    path.add(pr, "person")
    path.add(e)
    path.add(pl, "place")
    
    

    df = graph.traverse_path(
        paths=[path],
        conditions=[[f"person.id = {personId}"]],
        return_values=[
            "person.firstName", 
            " person.lastName",
        "person.birthday",
        "person.locationIP",
        "person.browserUsed",
            "place.name"],
        sort=[]
    )
    return df

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IS1_profile(graph, 1129)
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [44]:
#IS3

def IS3_friends(graph, personId: int):

    # (p:Person)-[:KNOWS]-(f:Person)

    p = Node(labels=[Label("Person")])
    f = Node(labels=[Label("Person")])
    k = Edge(p, f, label=Label("KNOWS"))

    path = Path()
    path.add(p, "person")
    path.add(k, "k")
    path.add(f, "friend")

    df = graph.traverse_path(
        paths=[path],
        conditions=[[f"person.id = {personId}"]],
        return_values=[
            "friend.id",
            "k.creationDate"
        ],
        sort=["friend.id"]
    )

    print(df) 
   

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IS3_friends(graph,94)
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [7]:
def IS4_message_content(graph, messageId: int):

    # Try Comment first
    comment_pattern = Node(
        labels=[Label("Comment")],
        properties=[Property("id", int, messageId)]
    )

    comment = graph.get_node(comment_pattern)

    if comment is not None:
        return (
            comment["content"],
            comment["creationDate"]
        )

    # Otherwise try Post
    post_pattern = Node(
        labels=[Label("Post")],
        properties=[Property("id", int, messageId)]
    )

    post = graph.get_node(post_pattern)

    if post is not None:
        return (
            post["content"],
            post["creationDate"]
        )

    return None

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IS4_message_content(graph,2061584302087)
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [49]:
def IS5_creator_of_message(graph, messageId: int):

    # ----Comment case ----------
    comment = Node(labels=[Label("Comment")])
    author = Node(labels=[Label("Person")])
    e1 = Edge(comment, author, label=Label("HAS_CREATOR"))

    path_comment = Path()
    path_comment.add(comment, "message")
    path_comment.add(e1)
    path_comment.add(author, "author")

    df = graph.traverse_path(
        paths=[path_comment],
        conditions=[[f"message.id = {messageId}"]],
        return_values=["author.id"],
        sort=[]
    )

    if not df.empty:
        return df


    # ---------Post case---------------------------------
    post = Node(labels=[Label("Post")])
    e2 = Edge(post, author, label=Label("HAS_CREATOR"))

    path_post = Path()
    path_post.add(post, "message")
    path_post.add(e2)
    path_post.add(author, "author")

    df = graph.traverse_path(
        paths=[path_post],
        conditions=[[f"message.id = {messageId}"]],
        return_values=["author.id"],
        sort=[]
    )

    return df

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IS5_creator_of_message(graph,618475290625)
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [9]:
def IS7_replies_of_message(graph, messageId: int):

    #detect message type

    comment_pattern = Node(
        labels=[Label("Comment")],
        properties=[Property("id", int, messageId)]
    )

    post_pattern = Node(
        labels=[Label("Post")],
        properties=[Property("id", int, messageId)]
    )

    message_node = graph.get_node(comment_pattern)

    if message_node is not None:
        message_label = "Comment"
    else:
        message_node = graph.get_node(post_pattern)
        if message_node is None:
            return None
        message_label = "Post"


    #STEP 1 : find replies

    reply = Node(labels=[Label("Comment")])
    message = Node(labels=[Label(message_label)])

    e_reply = Edge(reply, message, label=Label("REPLY_OF"))

    path_reply = Path()
    path_reply.add(reply, "reply")
    path_reply.add(e_reply)
    path_reply.add(message, "message")

    df = graph.traverse_path(
        paths=[path_reply],
        conditions=[[f"message.id = {messageId}"]],
        return_values=["reply.id"],
        sort=[]
    )

    if df.empty:
        return df


    #STEP 2 : message author

    message_pattern = Node(
        labels=[Label(message_label)],
        properties=[Property("id", int, messageId)]
    )

    messageAuthor = Node(labels=[Label("Person")])

    e_msg_creator = Edge(message_pattern, messageAuthor, label=Label("HAS_CREATOR"))

    path_msg_author = Path()
    path_msg_author.add(message_pattern, "message")
    path_msg_author.add(e_msg_creator)
    path_msg_author.add(messageAuthor, "messageAuthor")

    df_ma = graph.traverse_path(
        paths=[path_msg_author],
        return_values=["messageAuthor.id"]
    )

    if df_ma.empty:
        return None

    messageAuthorId = df_ma.iloc[0]["messageAuthor.id"]


    results = []


    # sTEP 3 : process each reply

    for _, row in df.iterrows():

        replyId = row["reply.id"]

        reply_node = Node(
            labels=[Label("Comment")],
            properties=[Property("id", int, replyId)]
        )

        replyAuthor = Node(labels=[Label("Person")])

        e_reply_creator = Edge(reply_node, replyAuthor, label=Label("HAS_CREATOR"))

        path_reply_author = Path()
        path_reply_author.add(reply_node, "reply")
        path_reply_author.add(e_reply_creator)
        path_reply_author.add(replyAuthor, "replyAuthor")

        df_ra = graph.traverse_path(
            paths=[path_reply_author],
            return_values=["replyAuthor.id"]
        )

        if df_ra.empty:
            continue

        replyAuthorId = df_ra.iloc[0]["replyAuthor.id"]


        #STEP 4 : compute KNOWS

        if replyAuthorId == messageAuthorId:
            knows = False

        else:

            p1 = Node(
                labels=[Label("Person")],
                properties=[Property("id", int, replyAuthorId)]
            )

            p2 = Node(
                labels=[Label("Person")],
                properties=[Property("id", int, messageAuthorId)]
            )

            e_knows = Edge(p1, p2, label=Label("KNOWS"))

            path_knows = Path()
            path_knows.add(p1, "p1")
            path_knows.add(e_knows)
            path_knows.add(p2, "p2")

            df_knows = graph.traverse_path(
                paths=[path_knows],
                return_values=["p1.id"]
            )

            knows = not df_knows.empty


        results.append(
            (
                replyId,
                replyAuthorId,
                messageAuthorId,
                knows
            )
        )

    return results

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IS7_replies_of_message(graph,1030792151049)
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [11]:
def IC1_transitive_friends(graph, personId: int, firstName: str):

    start = Node(labels=[Label("Person")])
    p1 = Node(labels=[Label("Person")])
    p2 = Node(labels=[Label("Person")])
    p3 = Node(labels=[Label("Person")])

    k1 = Edge(start, p1, label=Label("KNOWS"))
    k2 = Edge(p1, p2, label=Label("KNOWS"))
    k3 = Edge(p2, p3, label=Label("KNOWS"))

    results = set()

    # -------------------
    # distance = 1
    # -------------------

    path1 = Path()
    path1.add(start, "start")
    path1.add(k1)
    path1.add(p1, "person")

    df1 = graph.traverse_path(
        paths=[path1],
        conditions=[[
            f"start.id = {personId}",
            f"person.firstName = '{firstName}'",
            f"person.id <> {personId}"
        ]],
        return_values=["person.id"],
    )

    for _, row in df1.iterrows():
        results.add((int(row["person.id"]), 1))

    # -------------------
    # distance = 2
    # -------------------

    path2 = Path()
    path2.add(start, "start")
    path2.add(k1)
    path2.add(p1)
    path2.add(k2)
    path2.add(p2, "person")

    df2 = graph.traverse_path(
        paths=[path2],
        conditions=[[
            f"start.id = {personId}",
            f"person.firstName = '{firstName}'",
            f"person.id <> {personId}"
        ]],
        return_values=["person.id"],
    )

    for _, row in df2.iterrows():
        results.add((int(row["person.id"]), 2))

    # -------------------
    # distance = 3
    # -------------------

    path3 = Path()
    path3.add(start, "start")
    path3.add(k1)
    path3.add(p1)
    path3.add(k2)
    path3.add(p2)
    path3.add(k3)
    path3.add(p3, "person")

    df3 = graph.traverse_path(
        paths=[path3],
        conditions=[[
            f"start.id = {personId}",
            f"person.firstName = '{firstName}'",
            f"person.id <> {personId}"
        ]],
        return_values=["person.id"],
    )

    for _, row in df3.iterrows():
        results.add((int(row["person.id"]), 3))

    # -------------------
    # workplaces + studies
    # -------------------

    final = []

    for pid, dist in results:

        person = Node(
            labels=[Label("Person")],
            properties=[Property("id", int, pid)]
        )

        org = Node(labels=[Label("Organisation")])
        place = Node(labels=[Label("Place")])

        work = Edge(person, org, label=Label("WORK_AT"))
        study = Edge(person, org, label=Label("STUDY_AT"))
        loc = Edge(org, place, label=Label("IS_LOCATED_IN"))

        # workplaces
        path_work = Path()
        path_work.add(person, "p")
        path_work.add(work)
        path_work.add(org, "org")
       

        df_work = graph.traverse_path(
            paths=[path_work],
            conditions=[[f"p.id = {pid}"]],
            return_values=["org.name"]
        )

        # studies
        path_study = Path()
        path_study.add(person, "p")
        path_study.add(study)
        path_study.add(org, "org")
      

        df_study = graph.traverse_path(
            paths=[path_study],
            conditions=[[f"p.id = {pid}"]],
            return_values=["org.name"]
        )

        final.append({
            "personId": int(pid),
            "distance": dist,
            "workplaces": df_work.values.tolist(),
            "studies": df_study.values.tolist()
        })

    
    return final

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IC1_transitive_friends(graph, 2199023256684, "Boy")
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [ ]:
def IC3(graph, personId: int, countryXName: str, countryYName: str,
        startDate: int, durationDays: int):

# I need to retrieve friends and friends-of-friends (FOF) who created Comments or Posts in countries X and Y but do not live in those countries.

# To do this:
# 1. From the start, I retrieve friends and FOF who do not live in these countries; that is, I need to add the "isLocatedIn" edge in path1 and path2,
#    and add a condition that the country name is neither country X nor country Y.

# 2. Among those friends and FOF who do not live in these countries, I retrieve those who have published Comments or Posts in countries X and Y.

    endDate = startDate + durationDays

    # -------------------------------------------
    # STEP 1 : friends + friends of friendss
    # who DO NOT live in country X or Y ( only foreign persons)
    # ------------------------------------------------

    start = Node(
        labels=[Label("Person")],
        properties=[Property("id", int, personId)]
    )

    friend = Node(labels=[Label("Person")])
    fof = Node(labels=[Label("Person")])

    city = Node(labels=[Label("Place")])
    country = Node(labels=[Label("Place")])

    e1 = Edge(start, friend, label=Label("KNOWS"))
    e2 = Edge(friend, fof, label=Label("KNOWS"))

    e_loc1 = Edge(friend, city, label=Label("IS_LOCATED_IN"))
    e_loc2 = Edge(fof, city, label=Label("IS_LOCATED_IN"))

    e_part = Edge(city, country, label=Label("IS_PART_OF"))

    path_friend = Path()
    path_friend.add(start, "start")
    path_friend.add(e1)
    path_friend.add(friend, "person")
    path_friend.add(e_loc1)
    path_friend.add(city)
    path_friend.add(e_part)
    path_friend.add(country, "country")

    path_fof = Path()
    path_fof.add(start, "start")
    path_fof.add(e1)
    path_fof.add(friend)
    path_fof.add(e2)
    path_fof.add(fof, "person")
    path_fof.add(e_loc2)
    path_fof.add(city)
    path_fof.add(e_part)
    path_fof.add(country, "country")

    df_people = graph.traverse_path(
        paths=[path_friend, path_fof],
        conditions=[
            [
                f"start.id = {personId}",
                f"country.name <> '{countryXName}'",
                f"country.name <> '{countryYName}'"
            ],
            [
                f"start.id = {personId}",
                f"country.name <> '{countryXName}'",
                f"country.name <> '{countryYName}'"
            ]
        ],
        return_values=["person.id"]
    )

    if df_people.empty:
        return []

    person_ids = set(df_people["person.id"].dropna())

    # ------------------------------------------------
    # STEP 2 — for each foreign friends/fof,  retrieve messages created in the interval start date  & end date
    # ------------------------------------------------

    results = []

    for pid in person_ids:

        author = Node(
            labels=[Label("Person")],
            properties=[Property("id", int, pid)]
        )

        post = Node(labels=[Label("Post")])
        comment = Node(labels=[Label("Comment")])

        e_post = Edge(post, author, label=Label("HAS_CREATOR"))
        e_comment = Edge(comment, author, label=Label("HAS_CREATOR"))

        path_post = Path()
        path_post.add(post, "message")
        path_post.add(e_post)
        path_post.add(author, "person")

        path_comment = Path()
        path_comment.add(comment, "message")
        path_comment.add(e_comment)
        path_comment.add(author, "person")

        df_per_messages = graph.traverse_path(
            paths=[path_post, path_comment],
            conditions=[
                [
                    f"message.creationDate >= {startDate}",
                    f"message.creationDate < {endDate}"
                ],
                [
                    f"message.creationDate >= {startDate}",
                    f"message.creationDate < {endDate}"
                ]
            ],
            return_values=[
                "person.id",
                "message.id"
            ]
        )

        if df_per_messages.empty:
            continue

        countries_posted = set()

        # ------------------------------------------------
        # STEP 3 — check message country
        #-------------------------------------------
        # The objectives of my logic:

        # df_per_messages contains:
        # person.id | message.id

        # For each person.id, and for each of their message.id, we retrieve its country.
        # We keep only the messages located in country X or Y.
        # We filter df_per_messages using these messages.
        # ------------------------------------------------

         

        msg_country_list = []

        for _, row in df_per_messages.iterrows():

            mid = row["message.id"]

            message = Node(
                properties=[Property("id", int, mid)]
            )

            place = Node(labels=[Label("Place")])
            country = Node(labels=[Label("Place")])

            e_loc = Edge(message, place, label=Label("IS_LOCATED_IN"))
            e_part = Edge(place, country, label=Label("IS_PART_OF"))

            path_msg_country = Path()
            path_msg_country.add(message, "message")
            path_msg_country.add(e_loc)
            path_msg_country.add(place)
            path_msg_country.add(e_part)
            path_msg_country.add(country, "country")

            df_msg_in_selected_countries = graph.traverse_path(
                paths=[path_msg_country],
                conditions=[[f"message.id = {mid}",
                f"(country.name = '{countryXName}' OR country.name = '{countryYName}')"]],
                return_values=["message.id", "country.name"]
            )



        # -------------------------------
        # STEP 4 — filter df_per_messages using valid messages
        # ------------------------------------------------

        valid_message_ids = set(df_msg_in_selected_countries["message.id"])


        df_wanted_per = df_per_messages[
            df_per_messages["message.id"].isin(valid_message_ids)
        ]


        # ------------------------------------------------
        # STEP 5: collect wanted persons
        # -------------------------------------------

        results.extend(df_wanted_per["person.id"].tolist())

In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    IC3(graph, 1129, "China", "France", 1341100800000, 2)
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")